# Лабораторная работа № 2
## Колоночное хранение, партиционирование и бенчмаркинг Parquet

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование
**Профиль:** «Информатика и дополнительное образование (робототехника)»
**Этап конвейера:** `Collect → Store → Process`
**Среда:** Google Colab · PySpark 4.2.0 · `local[*]` · Parquet · Snappy / ZSTD / GZIP

---

### Цель работы

Освоить колоночное хранение телеметрии в формате Parquet, физическое партиционирование набора
и экспериментальное сравнение объёма и времени чтения CSV и Parquet. Понять, за счёт чего
возникает выигрыш и когда он исчезает.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в GitHub или Colab.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Три физические версии набора: CSV, Parquet, partitioned Parquet | 2 |
| 2 | Размеры, коэффициент сжатия, влияние энтропии и выбора кодека | 2 |
| 3 | Воспроизводимый бенчмарк чтения: ≥ 3 повтора, медиана, честное сравнение | 2 |
| 4 | Column pruning, partition pruning, проблема мелких файлов | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения — пользуйтесь `help()` и документацией Spark.
> Ячейки с `assert` — самопроверка: если проверка прошла, задание засчитано. Числовые значения
> времени зависят от runtime Colab, поэтому оценивается методика измерения, а не абсолютные цифры.

### Источники

* Balapriya C. `data-science-tutorials` — <https://github.com/balapriyac/data-science-tutorials>
  (`pyspark/pyspark_write_parquet.ipynb`)
* Piotr Szul. `spark-tutorial` — <https://github.com/piotrszul/spark-tutorial>
* Apache Parquet — <https://parquet.apache.org/docs/>
* Spark. Parquet Files — <https://spark.apache.org/docs/latest/sql-data-sources-parquet.html>

---
## Задание 0. Подготовка среды и данных

### Минимальные теоретические сведения

**Колоночное хранение.** Parquet размещает рядом значения одной колонки, а не одной записи.
Отсюда два следствия: однородные данные сжимаются в разы лучше, а запрос, которому нужны две
колонки из десяти, читает только эти две — это **column pruning**.

**Партиционирование.** `partitionBy("robot_id")` раскладывает данные по каталогам вида
`robot_id=robot-7/`. Запрос с фильтром по этой колонке пропускает лишние каталоги целиком —
это **partition pruning**. Значение ключа при этом физически не хранится в файлах: оно
закодировано в имени каталога.

**Метрики работы:**

коэффициент уменьшения объёма  $C = S_{CSV} / S_{Parquet}$

ускорение чтения  $Speedup = T_{CSV} / T_{Parquet}$

Результаты зависят от runtime Colab и файлового кэша, поэтому каждое измерение повторяется
не менее трёх раз, а итогом берётся **медиана** (она устойчивее среднего к единичному выбросу
при первом «холодном» чтении).

In [1]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.2.0 which is incompatible.


In [2]:
import shutil
import time
import statistics
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)

ROOT = Path("/content/lab02_storage")
if not Path("/content").exists():          # запуск вне Colab
    ROOT = Path("./lab02_storage")

CSV_DIR         = ROOT / "csv"
PARQUET_DIR     = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог хранения:", ROOT)

Spark version: 4.2.0
Каталог хранения: /content/lab02_storage


### Генерация телеметрии

Набор порождается детерминированно, внешний датасет не нужен. Обратите внимание на колонку
`payload`: это SHA-256 от идентификатора, то есть **высокоэнтропийная** строка без повторов.
Она добавлена намеренно — в части 2 мы посмотрим, что она делает со сжатием.

DataFrame кэшируется: иначе Spark пересчитает его заново при каждой из трёх записей.

In [3]:
N = 300_000

df = (
    spark.range(N)
    .withColumn("robot_id",    F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor",      F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms",       (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration",   F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload",     F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
    .cache()
)

print("Строк:", df.count())
df.printSchema()
df.show(3, truncate=False)

Строк: 300000
root
 |-- robot_id: string (nullable = true)
 |-- sensor: string (nullable = false)
 |-- ts_ms: long (nullable = false)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)

+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot_id|sensor|ts_ms        |temperature|vibration          |payload                                                         |
+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot-0 |imu   |1720000000000|40.0       |0.0                |5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|
|robot-1 |motor |1720000000100|40.08      |0.068617294923393  |6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|
|robot-2 |imu   |1720000000200|40.16      |0.13693862262231407|d4735e3a265e16eee03f59718b9b5d0301

---
# Часть 1. Три физические версии набора (2 балла)

Один и тот же DataFrame нужно записать тремя способами и дальше сравнивать их между собой.

## Упражнение 1.1. CSV

Запишите `df` в CSV в каталог `CSV_DIR`. Режим записи — `overwrite`, заголовок включён
(`option("header", True)`).

In [5]:
# TODO: df.write ... .csv(str(CSV_DIR))
df.write.mode("overwrite").option("header", True).csv(str(CSV_DIR))

files = sorted(p.name for p in CSV_DIR.iterdir())
print("Файлов в каталоге CSV:", len(files))
print(files[:5])
assert CSV_DIR.exists() and any(p.suffix == ".csv" for p in CSV_DIR.iterdir())

Файлов в каталоге CSV: 6
['._SUCCESS.crc', '.part-00000-d970b667-ab43-453e-9069-926de27d7880-c000.csv.crc', '.part-00001-d970b667-ab43-453e-9069-926de27d7880-c000.csv.crc', '_SUCCESS', 'part-00000-d970b667-ab43-453e-9069-926de27d7880-c000.csv']


> **Обратите внимание.** Spark пишет не один файл, а каталог: по одному файлу `part-*` на каждую
> partition плюс маркер `_SUCCESS`. Так устроены все распределённые форматы — единый файл нельзя
> писать параллельно с нескольких узлов.

## Упражнение 1.2. Parquet со сжатием Snappy

Запишите тот же `df` в Parquet в каталог `PARQUET_DIR`, явно указав кодек
`option("compression", "snappy")`.

In [6]:
# TODO: df.write ... .parquet(str(PARQUET_DIR))
df.write.mode("overwrite").option("compression", "snappy").parquet(str(PARQUET_DIR))

parts = [p.name for p in PARQUET_DIR.iterdir() if p.suffix == ".parquet"]
print("Файлов parquet:", len(parts))
print(parts[:3])
assert len(parts) > 0
assert all("snappy" in p for p in parts), "В имени файла должен быть виден кодек"

Файлов parquet: 2
['part-00000-068d1edd-c073-4f5f-bc77-dfd176453af3-c000.snappy.parquet', 'part-00001-068d1edd-c073-4f5f-bc77-dfd176453af3-c000.snappy.parquet']


## Упражнение 1.3. Partitioned Parquet

Запишите набор в `PARTITIONED_DIR` с физическим партиционированием по `robot_id`
(`partitionBy`). Затем выведите список созданных каталогов и проверьте, что их ровно 12 —
по числу роботов.

In [7]:
# TODO: df.write ... .partitionBy(...).parquet(str(PARTITIONED_DIR))
df.write.mode("overwrite").partitionBy("robot_id").parquet(str(PARTITIONED_DIR))

part_dirs = sorted(p.name for p in PARTITIONED_DIR.iterdir() if p.is_dir())
print("Каталогов:", len(part_dirs))
print(part_dirs[:4])

assert len(part_dirs) == 12
assert all(d.startswith("robot_id=") for d in part_dirs)

Каталогов: 12
['robot_id=robot-0', 'robot_id=robot-1', 'robot_id=robot-10', 'robot_id=robot-11']


## Упражнение 1.4

Прочитайте один файл из каталога `robot_id=robot-7` напрямую (`spark.read.parquet` по полному
пути к каталогу партиции) и посмотрите его схему. Есть ли в ней колонка `robot_id`?
Ответьте письменно, почему.

In [8]:
one_partition = spark.read.parquet(str(PARTITIONED_DIR / "robot_id=robot-7"))
one_partition.printSchema()

ANSWER_1_4 = """
Нет, колонки robot_id в схеме этого DataFrame нет.
Это происходит потому, что при использовании partitionBy Spark физически удаляет эту колонку из самих файлов Parquet, чтобы избежать дублирования данных и сэкономить дисковое пространство.
Значение колонки уже неявно закодировано в имени родительской директории (robot_id=robot-7), и при прямом чтении конкретной подпапки (а не корневого каталога) Spark не восстанавливает её в схеме.
"""
print(ANSWER_1_4)

assert "robot_id" not in one_partition.columns

root
 |-- sensor: string (nullable = true)
 |-- ts_ms: long (nullable = true)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)


Нет, колонки robot_id в схеме этого DataFrame нет. 
Это происходит потому, что при использовании partitionBy Spark физически удаляет эту колонку из самих файлов Parquet, чтобы избежать дублирования данных и сэкономить дисковое пространство. 
Значение колонки уже неявно закодировано в имени родительской директории (robot_id=robot-7), и при прямом чтении конкретной подпапки (а не корневого каталога) Spark не восстанавливает её в схеме.



---
# Часть 2. Объём хранения (2 балла)

## Упражнение 2.1. Размеры и коэффициент сжатия

Напишите функцию `dir_size(path)`, возвращающую суммарный размер всех файлов каталога в байтах
(каталог вложенный, поэтому нужен рекурсивный обход — подсказка: `Path.rglob`).

Выведите таблицу размеров трёх версий и рассчитайте коэффициент
$C = S_{CSV} / S_{Parquet}$.

In [9]:
def dir_size(path: Path) -> int:
    """Суммарный размер всех файлов внутри каталога, включая вложенные."""
    # TODO
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())
    raise NotImplementedError


sizes = {
    "CSV":                 dir_size(CSV_DIR),
    "Parquet":             dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}
for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:7.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"\nКоэффициент C = S_CSV / S_Parquet = {compression_ratio:.2f}x")

assert sizes["Parquet"] < sizes["CSV"], "Parquet должен быть компактнее CSV"
assert compression_ratio > 1.0

CSV                   :   33.96 MiB
Parquet               :   22.67 MiB
Partitioned Parquet   :   22.85 MiB

Коэффициент C = S_CSV / S_Parquet = 1.50x


## Упражнение 2.2. Энтропия и сжимаемость

Коэффициент из предыдущего упражнения получился скромным. Причина — колонка `payload`:
это SHA-256, то есть практически случайная строка. Данные с высокой энтропией не сжимаются
принципиально: в них нет повторов, которые можно было бы закодировать короче.

Запишите **тот же набор без колонки `payload`** в CSV и в Parquet, посчитайте коэффициент
заново и сравните с предыдущим.

In [11]:
NOPAY_CSV = ROOT / "csv_nopayload"
NOPAY_PQ  = ROOT / "parquet_nopayload"

df_nopay = df.drop("payload")

# TODO: запишите df_nopay в CSV (с заголовком) и в Parquet (snappy)
df_nopay.write.mode("overwrite").option("header", True).csv(str(NOPAY_CSV))
df_nopay.write.mode("overwrite").option("compression", "snappy").parquet(str(NOPAY_PQ))

sizes_nopay = {"CSV": dir_size(NOPAY_CSV), "Parquet": dir_size(NOPAY_PQ)}
ratio_nopay = sizes_nopay["CSV"] / sizes_nopay["Parquet"]

print(f"Без payload: CSV {sizes_nopay['CSV']/1024/1024:.2f} MiB, "
      f"Parquet {sizes_nopay['Parquet']/1024/1024:.2f} MiB")
print(f"Коэффициент без payload: {ratio_nopay:.2f}x  (с payload было {compression_ratio:.2f}x)")

ANSWER_2_2 = """
Коэффициент сжатия значительно вырос, потому что мы удалили колонку payload с SHA-256 хешами, которая обладает высокой энтропией (это практически случайные данные).
Алгоритмы сжатия (как Snappy в Parquet, так и стандартные для CSV) работают за счет поиска повторяющихся паттернов, которых почти нет в случайных строках, из-за чего они сжимаются очень плохо.
Оставшиеся колонки содержат структурированные данные с низкой энтропией (числа, даты, повторяющиеся ID), которые Parquet сжимает гораздо эффективнее.
Благодаря колоночному формату хранения и дополнительным методам кодирования (например, dictionary encoding для повторяющихся ID), Parquet выигрывает у CSV в размере намного сильнее, когда в данных нет "шума".
"""
print(ANSWER_2_2)

assert ratio_nopay > compression_ratio, "Без высокоэнтропийной колонки выигрыш должен вырасти"

Без payload: CSV 15.22 MiB, Parquet 3.61 MiB
Коэффициент без payload: 4.21x  (с payload было 1.50x)

Коэффициент сжатия значительно вырос, потому что мы удалили колонку payload с SHA-256 хешами, которая обладает высокой энтропией (это практически случайные данные). 
Алгоритмы сжатия (как Snappy в Parquet, так и стандартные для CSV) работают за счет поиска повторяющихся паттернов, которых почти нет в случайных строках, из-за чего они сжимаются очень плохо. 
Оставшиеся колонки содержат структурированные данные с низкой энтропией (числа, даты, повторяющиеся ID), которые Parquet сжимает гораздо эффективнее. 
Благодаря колоночному формату хранения и дополнительным методам кодирования (например, dictionary encoding для повторяющихся ID), Parquet выигрывает у CSV в размере намного сильнее, когда в данных нет "шума".



## Упражнение 2.3. Сравнение кодеков

Запишите `df_nopay` в Parquet четырьмя кодеками: `uncompressed`, `snappy`, `gzip`, `zstd`.
Для каждого измерьте время записи и итоговый размер. Заполните словарь `codec_stats`
в формате `{кодек: (секунды, байты)}`.

In [13]:
codec_stats = {}

for codec in ["uncompressed", "snappy", "gzip", "zstd"]:
    out = ROOT / f"pq_{codec}"
    # TODO: замерьте время записи df_nopay с этим кодеком и размер каталога
    start_time = time.time()

    df_nopay.write.mode("overwrite").option("compression", codec).parquet(str(out))

    end_time = time.time()
    duration = end_time - start_time

    size = dir_size(out)

    codec_stats[codec] = (duration, size)


print(f"{'кодек':<14}{'время, с':>10}{'размер, MiB':>14}{'C к uncompressed':>20}")
base = codec_stats["uncompressed"][1]
for codec, (t, s) in codec_stats.items():
    print(f"{codec:<14}{t:>10.2f}{s/1024/1024:>14.2f}{base/s:>20.2f}")

assert set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"}
assert codec_stats["gzip"][1] < codec_stats["snappy"][1], "gzip должен сжимать сильнее snappy"

кодек           время, с   размер, MiB    C к uncompressed
uncompressed        1.22          5.09                1.00
snappy              1.26          3.61                1.41
gzip                1.34          2.70                1.89
zstd                0.60          2.61                1.95


**Вопрос 2.4.** Почему Snappy используется по умолчанию, хотя gzip сжимает заметно сильнее?
Ответьте письменно.

In [14]:
ANSWER_2_4 = """
Snappy используется по умолчанию, потому что в распределенных системах (как Spark) критически важна общая скорость выполнения запроса, а не только экономия места на диске.
Хотя gzip обеспечивает лучшую степень сжатия, он требует значительных затрат процессорного времени (CPU) на распаковку, что часто становится "узким местом" (bottleneck) при чтении больших объемов данных.
Snappy же разработан для экстремально высокой скорости декомпрессии: он жертвует частью коэффициента сжатия ради того, чтобы данные распаковывались практически со скоростью чтения с диска или из оперативной памяти.
В результате общее время чтения и обработки данных с Snappy оказывается меньше, чем с gzip, что делает его идеальным компромиссом (trade-off) для задач Big Data.
"""
print(ANSWER_2_4)


Snappy используется по умолчанию, потому что в распределенных системах (как Spark) критически важна общая скорость выполнения запроса, а не только экономия места на диске. 
Хотя gzip обеспечивает лучшую степень сжатия, он требует значительных затрат процессорного времени (CPU) на распаковку, что часто становится "узким местом" (bottleneck) при чтении больших объемов данных. 
Snappy же разработан для экстремально высокой скорости декомпрессии: он жертвует частью коэффициента сжатия ради того, чтобы данные распаковывались практически со скоростью чтения с диска или из оперативной памяти. 
В результате общее время чтения и обработки данных с Snappy оказывается меньше, чем с gzip, что делает его идеальным компромиссом (trade-off) для задач Big Data.



---
# Часть 3. Бенчмарк чтения (2 балла)

## Упражнение 3.1. Функция измерения

Напишите функцию `benchmark_read(fmt, path, predicate=None, repeats=3, schema=None)`, которая:

1. `repeats` раз читает набор из `path` в формате `fmt` (`"csv"` или `"parquet"`);
2. при заданном `predicate` применяет фильтр;
3. выполняет действие `count()` — **без него из-за ленивости ничего не прочитается**;
4. возвращает кортеж `(медиана времени, число строк)`.

Для CSV используйте `option("header", True)`; параметр `schema` пока оставьте
необязательным — он понадобится в упражнении 3.3.

In [15]:
def benchmark_read(fmt, path, predicate=None, repeats=3, schema=None):
    """Возвращает (медианное время чтения, число строк)."""
    times, counts = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        # TODO: прочитайте данные нужного формата, примените predicate, вызовите count()

        reader = spark.read
        if schema is not None:
            reader = reader.schema(schema)
        if fmt == "csv":
            reader = reader.option("header", True)

        df = reader.format(fmt).load(str(path))

        if predicate is not None:
            df = df.filter(predicate)

        n = df.count()

        t1 = time.perf_counter()
        times.append(t1 - t0)
        counts.append(n)
    return statistics.median(times), counts[-1]


t, n = benchmark_read("parquet", PARQUET_DIR, repeats=3)
print(f"Чтение всего Parquet: {t:.3f} с, строк {n}")
assert n == N

Чтение всего Parquet: 0.407 с, строк 300000


## Упражнение 3.2. Сравнение трёх версий

Выполните одинаковый запрос `robot_id == "robot-7"` к каждой из трёх версий набора, минимум
по три повтора. Рассчитайте ускорение относительно CSV.

In [16]:
predicate = F.col("robot_id") == "robot-7"

csv_t,  csv_n  = benchmark_read("csv",     CSV_DIR,         predicate)
pq_t,   pq_n   = benchmark_read("parquet", PARQUET_DIR,     predicate)
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, predicate)

print(f"CSV                 : {csv_t:6.3f} с, строк {csv_n}")
print(f"Parquet             : {pq_t:6.3f} с, строк {pq_n}")
print(f"Partitioned Parquet : {part_t:6.3f} с, строк {part_n}")
print()
print(f"Speedup CSV -> Parquet             : {csv_t / pq_t:.2f}x")
print(f"Speedup CSV -> Partitioned Parquet : {csv_t / part_t:.2f}x")

assert csv_n == pq_n == part_n == N // 12, "Все три версии должны вернуть одинаковое число строк"

CSV                 :  3.220 с, строк 25000
Parquet             :  0.405 с, строк 25000
Partitioned Parquet :  0.353 с, строк 25000

Speedup CSV -> Parquet             : 7.96x
Speedup CSV -> Partitioned Parquet : 9.12x


## Упражнение 3.3. Честное сравнение

В упражнении 3.1 CSV читался с `inferSchema=True`. Это удобно, но нечестно по отношению к CSV:
чтобы вывести типы, Spark делает **дополнительный полный проход** по файлу. Получается, что
мы сравниваем Parquet с CSV, прочитанным дважды.

Повторите замер CSV с **явно заданной схемой** — возьмите её у уже прочитанного Parquet
(`spark.read.parquet(...).schema`) — и сравните три числа: CSV с выводом типов, CSV со схемой
и Parquet.

In [18]:
explicit_schema = spark.read.parquet(str(PARQUET_DIR)).schema

# TODO: замерьте чтение CSV с явной схемой (schema=explicit_schema)
csv_schema_t, csv_schema_n = benchmark_read("csv", CSV_DIR, schema=explicit_schema, repeats=3)

print(f"CSV + inferSchema : {csv_t:6.3f} с")
print(f"CSV + явная схема : {csv_schema_t:6.3f} с")
print(f"Parquet           : {pq_t:6.3f} с")
print(f"\nЧестный speedup CSV -> Parquet: {csv_schema_t / pq_t:.2f}x")

ANSWER_3_3 = """
TODO: насколько изменился выигрыш Parquet и почему такое уточнение важно?
Выигрыш Parquet над CSV уменьшился, потому что мы убрали накладные расходы на inferSchema — дополнительный полный проход Spark по CSV-файлу для определения типов колонок.
Без этого прохода CSV читается заметно быстрее, и разница с Parquet сокращается.
Это уточнение важно, потому что в реальных production-системах схему CSV почти всегда задают явно (чтобы избежать некорректного вывода типов и лишних проходов), поэтому «нечестное» сравнение с inferSchema завышает преимущество Parquet.
Тем не менее Parquet всё равно быстрее благодаря колоночному хранению, встроенной схеме в метаданных и эффективному сжатию — просто реальный speedup скромнее, чем казалось на первый взгляд.
"""
print(ANSWER_3_3)

assert csv_schema_n == csv_n
assert csv_schema_t < csv_t, "Без inferSchema чтение CSV должно ускориться"

CSV + inferSchema :  3.220 с
CSV + явная схема :  0.439 с
Parquet           :  0.405 с

Честный speedup CSV -> Parquet: 1.08x

TODO: насколько изменился выигрыш Parquet и почему такое уточнение важно?
Выигрыш Parquet над CSV уменьшился, потому что мы убрали накладные расходы на inferSchema — дополнительный полный проход Spark по CSV-файлу для определения типов колонок. 
Без этого прохода CSV читается заметно быстрее, и разница с Parquet сокращается. 
Это уточнение важно, потому что в реальных production-системах схему CSV почти всегда задают явно (чтобы избежать некорректного вывода типов и лишних проходов), поэтому «нечестное» сравнение с inferSchema завышает преимущество Parquet. 
Тем не менее Parquet всё равно быстрее благодаря колоночному хранению, встроенной схеме в метаданных и эффективному сжатию — просто реальный speedup скромнее, чем казалось на первый взгляд.



AssertionError: 

---
# Часть 4. Pruning и мелкие файлы (2 балла)

## Упражнение 4.1. Partition pruning

Выполните `explain(mode="formatted")` для запроса к partitioned Parquet с фильтром по
`robot_id` и найдите в плане строку `PartitionFilters`.

In [19]:
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [temperature#2507, robot_id#2510]
Batched: true
Location: InMemoryFileIndex [file:/content/lab02_storage/parquet_partitioned]
PartitionFilters: [isnotnull(robot_id#2510), (robot_id#2510 = robot-7)]
ReadSchema: struct<temperature:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [temperature#2507, robot_id#2510]

(3) Project [codegen id : 1]
Output [2]: [robot_id#2510, temperature#2507]
Input [2]: [temperature#2507, robot_id#2510]




In [20]:
ANSWER_4_1 = """
TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
PartitionFilters содержит условия фильтрации, которые применяются к путям (именам директорий) ДО физического чтения файлов. В данном случае это фильтр по robot_id. Это позволяет механизму Spark пропустить целые папки с данными (partition pruning), не тратя время на их открытие.
PushedFilters — это фильтры, которые "проталкиваются" (push down) непосредственно в источник данных (Parquet reader) и применяются к строкам внутри уже прочитанных файлов (например, с использованием встроенной статистики min/max). В этом запросе PushedFilters пуст, так как фильтр по robot_id уже полностью обработан на уровне партиций.
ReadSchema показывает, какие именно колонки физически считываются с диска. Здесь ReadSchema содержит только temperature, потому что колонка robot_id не хранится в самих файлах Parquet, а виртуально восстанавливается из имен родительских директорий (например, robot_id=robot-7).
"""
print(ANSWER_4_1)


TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
PartitionFilters содержит условия фильтрации, которые применяются к путям (именам директорий) ДО физического чтения файлов. В данном случае это фильтр по robot_id. Это позволяет механизму Spark пропустить целые папки с данными (partition pruning), не тратя время на их открытие.
PushedFilters — это фильтры, которые "проталкиваются" (push down) непосредственно в источник данных (Parquet reader) и применяются к строкам внутри уже прочитанных файлов (например, с использованием встроенной статистики min/max). В этом запросе PushedFilters пуст, так как фильтр по robot_id уже полностью обработан на уровне партиций.
ReadSchema показывает, какие именно колонки физически считываются с диска. Здесь ReadSchema содержит только temperature, потому что колонка robot_id не хранится в самих файлах Parquet, а виртуально восстанавливается из имен родительских директорий (например, robot

## Упражнение 4.2. Column pruning в цифрах

Сравните время двух запросов к обычному (непартиционированному) Parquet:

* агрегация **только** по `temperature`;
* агрегация по `temperature` **и** по тяжёлой высокоэнтропийной колонке `payload`.

Второй запрос вынуждает Spark поднять с диска колонку, которая занимает большую часть объёма
файла. Каждый запрос — минимум три повтора, берите медиану.

Дополнительно выведите `explain` обоих вариантов и сравните строки `ReadSchema`.

In [21]:
def timed(fn, repeats=3):
    """Медианное время выполнения fn()."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


def read_with_payload():
    # TODO: agg по temperature И по payload (например, F.max("payload"))
    spark.read.parquet(str(PARQUET_DIR)) \
        .agg(F.avg("temperature"), F.max("payload")) \
        .collect()


def read_one_column():
    # TODO: agg только по temperature
    spark.read.parquet(str(PARQUET_DIR)) \
        .agg(F.avg("temperature")) \
        .collect()


t_all = timed(read_with_payload)
t_one = timed(read_one_column)
print(f"temperature + payload: {t_all:.3f} с")
print(f"только temperature   : {t_one:.3f} с")
print(f"Column pruning дал ускорение в {t_all / t_one:.2f}x")

spark.read.parquet(str(PARQUET_DIR)).select("temperature").agg(F.avg("temperature")).explain()

ANSWER_4_2 = """
TODO: ваш ответ
Column pruning (отсечение колонок) дал значительное ускорение, потому что Parquet — это колоночный формат хранения.
Когда запрос требует только колонку temperature, Parquet-ридер физически считывает с диска и распаковывает только блоки данных, относящиеся к этой колонке, полностью игнорируя тяжелую колонку payload.
Поскольку payload состоит из длинных SHA-256 строк с высокой энтропией, она занимает большую часть объема файла.
Избегая чтения и декомпрессии этого "мусорного" объема данных, Spark радикально снижает нагрузку на дисковый ввод-вывод (I/O) и процессор (CPU), что и дает кратный выигрыш в скорости.
"""
print(ANSWER_4_2)

temperature + payload: 1.724 с
только temperature   : 0.561 с
Column pruning дал ускорение в 3.07x
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(temperature#2671)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1274]
      +- HashAggregate(keys=[], functions=[partial_avg(temperature#2671)])
         +- FileScan parquet [temperature#2671] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab02_storage/parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<temperature:double>



TODO: ваш ответ
Column pruning (отсечение колонок) дал значительное ускорение, потому что Parquet — это колоночный формат хранения. 
Когда запрос требует только колонку temperature, Parquet-ридер физически считывает с диска и распаковывает только блоки данных, относящиеся к этой колонке, полностью игнорируя тяжелую колонку payload. 
Поскольку payload состоит из длинных SHA-256 строк 

## Упражнение 4.3. Проблема мелких файлов

Партиционирование выглядит бесплатным улучшением, но это не так. Запишите набор,
партиционированный по искусственному ключу высокой кардинальности `ts_bucket` (200 значений),
и сравните с партиционированием по `robot_id` (12 значений):

* число созданных файлов;
* суммарный объём;
* средний размер одного файла;
* время чтения всего набора.

In [22]:
MANY_DIR = ROOT / "parquet_many_partitions"

df_buckets = df_nopay.withColumn(
    "ts_bucket", ((F.col("ts_ms") / 100) % 200).cast("int")
)
# TODO: запишите df_buckets в MANY_DIR с partitionBy("ts_bucket")
df_buckets.write.mode("overwrite").partitionBy("ts_bucket").parquet(str(MANY_DIR))

def file_stats(path: Path):
    """Возвращает (число файлов .parquet, суммарный размер, средний размер)."""
    # TODO
    files = list(path.rglob("*.parquet"))
    n_files = len(files)

    if n_files == 0:
        return 0, 0, 0.0

    total_size = sum(p.stat().st_size for p in files)
    avg_size = total_size / n_files

    return n_files, total_size, avg_size


n_few,  s_few,  avg_few  = file_stats(PARTITIONED_DIR)
n_many, s_many, avg_many = file_stats(MANY_DIR)

print(f"{'вариант':<28}{'файлов':>9}{'объём, MiB':>13}{'средний файл, KiB':>20}")
print(f"{'partitionBy(robot_id) [12]':<28}{n_few:>9}{s_few/1024/1024:>13.2f}{avg_few/1024:>20.1f}")
print(f"{'partitionBy(ts_bucket) [200]':<28}{n_many:>9}{s_many/1024/1024:>13.2f}{avg_many/1024:>20.1f}")

t_few  = timed(lambda: spark.read.parquet(str(PARTITIONED_DIR)).count())
t_many = timed(lambda: spark.read.parquet(str(MANY_DIR)).count())
print(f"\nЧтение целиком: 12 партиций {t_few:.3f} с, 200 партиций {t_many:.3f} с")

assert n_many > n_few
assert avg_many < avg_few

вариант                        файлов   объём, MiB   средний файл, KiB
partitionBy(robot_id) [12]         24        22.67               967.5
partitionBy(ts_bucket) [200]      400         4.47                11.4

Чтение целиком: 12 партиций 0.911 с, 200 партиций 7.479 с


## Упражнение 4.4. Технический вывод

Ответьте письменно:

1. Почему слишком большое число мелких partition ухудшает эффективность хранения и чтения?
2. По какому признаку выбирать колонку для `partitionBy`?
3. Когда партиционирование по `robot_id` в нашей задаче окажется бесполезным?

In [23]:
ANSWER_4_4 = """
TODO: ваш ответ (7-10 предложений)
Слишком большое число мелких файлов (проблема Small Files) создает огромные накладные расходы на уровне файловой системы и Spark-драйвера, которому приходится хранить метаданные для тысяч директорий и открывать каждый файл отдельно.
Кроме того, Spark вынужден планировать и запускать отдельную задачу (task) для чтения каждого мелкого файла, из-за чего время на оркестрацию начинает превышать время самого чтения данных.
Колонку для partitionBy следует выбирать по двум главным критериям: она должна иметь низкую кардинальность (немного уникальных значений, чтобы не плодить тысячи файлов) и часто использоваться в фильтрах, чтобы работал механизм отсечения партиций (partition pruning).
Идеальный размер одной партиции на диске должен составлять от 128 МБ до 1 ГБ, чтобы обеспечить эффективное последовательное чтение.
Партиционирование по robot_id в нашей задаче окажется совершенно бесполезным, если аналитические запросы не предполагают фильтрации по этому полю (например, мы всегда читаем данные всех роботов целиком для глобальной агрегации).
В таком случае Spark всё равно придется сканировать все 12 директорий, но при этом мы получим дополнительные накладные расходы на чтение метаданных и обработку множества файлов, что только замедлит выполнение запроса.
Таким образом, физическое разбиение данных имеет смысл только тогда, когда паттерны чтения данных строго соответствуют схеме партиционирования.
"""
print(ANSWER_4_4)


TODO: ваш ответ (7-10 предложений)
Слишком большое число мелких файлов (проблема Small Files) создает огромные накладные расходы на уровне файловой системы и Spark-драйвера, которому приходится хранить метаданные для тысяч директорий и открывать каждый файл отдельно.
Кроме того, Spark вынужден планировать и запускать отдельную задачу (task) для чтения каждого мелкого файла, из-за чего время на оркестрацию начинает превышать время самого чтения данных. 
Колонку для partitionBy следует выбирать по двум главным критериям: она должна иметь низкую кардинальность (немного уникальных значений, чтобы не плодить тысячи файлов) и часто использоваться в фильтрах, чтобы работал механизм отсечения партиций (partition pruning). 
Идеальный размер одной партиции на диске должен составлять от 128 МБ до 1 ГБ, чтобы обеспечить эффективное последовательное чтение. 
Партиционирование по robot_id в нашей задаче окажется совершенно бесполезным, если аналитические запросы не предполагают фильтрации по этому 

---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы.

**Требования к заполнению.** Формулировки конкретные. «Расскажу про Parquet» не засчитывается.
Нужны измеримый результат, критерий успеха и честно названные ограничения.

**Подсказка по содержанию.** Эксперимент «CSV против Parquet» удобен для школы тем, что результат
виден глазами: два каталога с одними и теми же данными занимают разный объём, и это можно
измерить без единой строки распределённого кода — хватит `du -h`. Аналогия для урока: если
нужна только температура, читается только «стопка температур», а не все листы журнала целиком.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | Архивариус данных роботов: почему Parquet быстрее и меньше? |
| **Целевая аудитория** | СПО |
| **Исследовательский вопрос** | Как выбор формата хранения (текстовый CSV против колоночного Parquet) и стратегии разбиения данных (партиционирование) влияют на объем занимаемого дискового пространства и время выполнения операций ввода-вывода (I/O) при ограниченных вычислительных ресурсах? |
| **Источник данных** | Синтетический датасет телеметрии устройств (ID устройства, timestamp, показания датчиков, хеш-сумма). Допускается замена на реальные логи веб-сервера (Nginx/Apache) или данные с датчиков Arduino/Raspberry Pi для повышения практической ценности. |
| **Инженерная концепция, сохраняемая без упрощения** | Принципы колоночного хранения данных; алгоритмы сжатия без потерь (Snappy/GZIP); физическое партиционирование (создание иерархии директорий); механизм отсечения партиций (partition pruning); проблема мелких файлов (Small Files Problem). |
| **Что упрощается относительно университетской лабораторной** | Не разворачивается полноценный распределенный кластер (Hadoop/Spark Cluster) — используется локальный режим (local[*]). Исключается тонкая настройка JVM, управление памятью (Garbage Collection) и сложные ETL-пайплайны очистки "грязных" данных. Данные предоставляются в структурированном виде, чтобы сфокусироваться именно на анализе производительности хранения. |
| **Алгоритм действий обучающегося (4–6 шагов)** | 1. Базовая запись: Сохранить идентичный набор данных в форматы CSV и Parquet. 2. Замер дискового пространства: Измерить физический размер полученных директорий (через du -h в Linux или свойства папки в Windows) и рассчитать коэффициент сжатия. 3. Анализ энтропии: Исключить из данных высокоэнтропийную колонку (например, случайный хеш), повторить запись и замерить, как изменился коэффициент сжатия. 4. Честный бенчмарк скорости: Замерить медианное время (по 3 запускам) выполнения операции .count() для CSV (с явной схемой) и Parquet. 5. Проверка Column Pruning: Замерить время чтения только одной колонки из Parquet и сравнить его со временем чтения всех колонок. 6. Анализ партиционирования: Разбить данные по папкам (например, по ID устройства). Замерить время чтения данных одного устройства и проанализировать план выполнения (explain()), найдя строку PartitionFilters. |
| **Измеримый результат / метрика** | Коэффициент сжатия: отношение размера CSV к размеру Parquet (числовое значение, например, 3.5x). Дельта времени: время выполнения запросов в миллисекундах (медиана по 3 запускам). Статистика файлов: общее количество файлов .parquet и их средний размер в КБ при разной глубине партиционирования. |
| **Критерий успешного выполнения** | 1. Предоставлен отчет (ноутбук или документ) с заполненными таблицами замеров. 2. Эмпирически доказано: Parquet занимает меньше места, а чтение конкретной колонки из Parquet быстрее чтения всего файла. 3. Студент может аргументированно объяснить (устно или письменно), почему партиционирование по полю с высокой кардинальностью (например, по секундам времени) ухудшит производительность системы. |
| **Вариант усложнения для НТО / хакатона** | Сравнить 4 разных кодека сжатия (uncompressed, snappy, gzip, zstd) и построить график зависимости «Время записи vs Итоговый размер». Реализовать тот же бенчмарк на чистом Python (связка pandas + pyarrow) и сравнить производительность с PySpark. Намеренно создать «проблему мелких файлов» (сгенерировать 1000+ партиций) и зафиксировать деградацию производительности или ошибку нехватки памяти. |
| **Риски и ограничения** | Кэширование диска: На малых объемах данных (< 50 МБ) разница во времени чтения будет статистически незначима из-за быстрого SSD и кэша ОС. Для чистоты эксперимента требуется датасет от 100–500 МБ. Ловушка ленивых вычислений: Студенты могут забыть вызвать «действие» (action, например, .count() или .collect()). В этом случае замерится только время построения плана выполнения (доли миллисекунды), а не реальное чтение с диска. Аппаратурные ограничения: Генерация тысяч мелких файлов на учебных компьютерах со слабым HDD или малым объемом RAM может привести к зависанию файловой системы или среды выполнения (Jupyter/Colab). |

---
## Итоговая самопроверка

In [24]:
checks = {
    "1.1 CSV записан":          CSV_DIR.exists(),
    "1.2 Parquet записан":      len(list(PARQUET_DIR.glob("*.parquet"))) > 0,
    "1.3 partitionBy":          len(part_dirs) == 12,
    "1.4 схема партиции":       "TODO" not in ANSWER_1_4,
    "2.1 размеры и C":          compression_ratio > 1.0,
    "2.2 энтропия":             ratio_nopay > compression_ratio and "TODO" not in ANSWER_2_2,
    "2.3 кодеки":               set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"},
    "2.4 выбор кодека":         "TODO" not in ANSWER_2_4,
    "3.2 бенчмарк чтения":      csv_n == pq_n == part_n,
    "3.3 честное сравнение":    csv_schema_t < csv_t and "TODO" not in ANSWER_3_3,
    "4.1 partition pruning":    "TODO" not in ANSWER_4_1,
    "4.2 column pruning":       t_all > t_one and "TODO" not in ANSWER_4_2,
    "4.3 мелкие файлы":         n_many > n_few,
    "4.4 технический вывод":    "TODO" not in ANSWER_4_4,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

OK   1.1 CSV записан
OK   1.2 Parquet записан
OK   1.3 partitionBy
OK   1.4 схема партиции
OK   2.1 размеры и C
OK   2.2 энтропия
OK   2.3 кодеки
OK   2.4 выбор кодека
OK   3.2 бенчмарк чтения
НЕТ  3.3 честное сравнение
НЕТ  4.1 partition pruning
НЕТ  4.2 column pruning
OK   4.3 мелкие файлы
НЕТ  4.4 технический вывод

Выполнено: 10 из 14
TEACH CARD проверяется преподавателем вручную.


In [25]:
df.unpersist()
spark.stop()
print("SparkSession остановлена.")

SparkSession остановлена.
